In [1]:
import pandas as pd
import numpy as np
import io
import pymc as pm
import arviz as az

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
# 1. 파일 업로드
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

if file_name.endswith('.xlsx') or file_name.endswith('.xls'):
    df = pd.read_excel(io.BytesIO(uploaded[file_name]))
else:
    df = pd.read_csv(io.BytesIO(uploaded[file_name]), encoding='cp949')

df.columns = df.columns.astype(str).str.strip()

print("원본 데이터 모양:", df.shape)
print("컬럼명:", df.columns.tolist())

Saving 사고분석-서울.xlsx to 사고분석-서울.xlsx
원본 데이터 모양: (1281, 22)
컬럼명: ['구분번호', '발생년월', '주야', '시군구', '사고내용', '사망자수', '중상자수', '경상자수', '부상신고자수', '사고유형', '법규위반', '노면상태', '기상상태', '도로형태', '가해운전자 차종', '가해운전자 성별', '가해운전자 연령대', '가해운전자 상해정도', '피해운전자 차종', '피해운전자 성별', '피해운전자 연령대', '피해운전자 상해정도']


In [3]:
# 2. 전처리

df['year'] = df['발생년월'].astype(str).str.extract(r'(\d{4})').astype(int)
df['month'] = df['발생년월'].astype(str).str.extract(r'(\d{1,2})월').astype(int)

df['District'] = df['시군구'].apply(
    lambda x: str(x).split()[1] if len(str(x).split()) > 1 else 'Unknown'
)

# 피해운전자 상해정도를 위험도 점수로 변환
# 현재 데이터에 사망이 없더라도 mapping은 남겨둔다.
injury_score_map = {
    '상해없음': 0,
    '부상신고': 25,
    '경상': 50,
    '중상': 75,
    '사망': 100,
    '기타불명': np.nan
}

df['injury_score'] = df['피해운전자 상해정도'].astype(str).map(injury_score_map)

# target 없는 행 제거
df = df.dropna(subset=['injury_score']).copy()

# 0~100 점수를 0~1 risk로 변환
df['risk'] = df['injury_score'] / 100

print("\n--- 피해운전자 상해정도 분포 ---")
print(df['피해운전자 상해정도'].value_counts(dropna=False))

print("\n--- risk 분포 ---")
print(df['risk'].value_counts().sort_index())


--- 피해운전자 상해정도 분포 ---
피해운전자 상해정도
경상      568
상해없음    336
중상      243
부상신고     75
사망        1
Name: count, dtype: int64

--- risk 분포 ---
risk
0.00    336
0.25     75
0.50    568
0.75    243
1.00      1
Name: count, dtype: int64


In [4]:
# 3. 사용할 변수 정의 및 인코딩

cat_cols = [
    'District',
    '주야',
    '기상상태',
    '노면상태',
    '도로형태',
    '사고유형',
    '법규위반',
    '가해운전자 성별',
    '가해운전자 연령대'
]

cat_cols = [c for c in cat_cols if c in df.columns]

encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    df[col + '_idx'] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

df['month_idx'] = df['month'] - 1

# train/test split
# risk 값 기준 stratify가 가능하면 적용
stratify_col = df['injury_score']

if stratify_col.value_counts().min() >= 2:
    stratify_option = stratify_col
else:
    stratify_option = None

train_idx, test_idx = train_test_split(
    np.arange(len(df)),
    test_size=0.2,
    stratify=stratify_option,
    random_state=42
)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

print("Train:", train.shape)
print("Test:", test.shape)

Train: (978, 37)
Test: (245, 37)


In [6]:
# 4. Beta regression을 위한 target 변환
# Beta distribution은 0과 1을 직접 다룰 수 없으므로 epsilon 처리

epsilon = 1e-3

train_risk = train['risk'].values
test_risk = test['risk'].values

train_risk_beta = np.clip(train_risk, epsilon, 1 - epsilon)
test_risk_beta = np.clip(test_risk, epsilon, 1 - epsilon)

In [7]:
# 5. Hierarchical Bayesian Beta Regression Model

coords = {
    "district": encoders['District'].classes_,
    "daynight": encoders['주야'].classes_,
    "weather": encoders['기상상태'].classes_,
    "surface": encoders['노면상태'].classes_,
    "roadtype": encoders['도로형태'].classes_,
    "accident_type": encoders['사고유형'].classes_,
    "violation": encoders['법규위반'].classes_,
    "gender": encoders['가해운전자 성별'].classes_,
    "age": encoders['가해운전자 연령대'].classes_,
    "month": np.arange(1, 13)
}

with pm.Model(coords=coords) as hb_risk_model:

    # 전체 평균 위험도
    alpha = pm.Normal("alpha", mu=0, sigma=1.5)

    # Beta 분포의 concentration parameter
    # 클수록 mu 주변에 target이 조밀하게 모임
    phi = pm.Exponential("phi", lam=1)

    # Non-centered hierarchical effects
    sigma_district = pm.HalfNormal("sigma_district", sigma=0.5)
    district_offset = pm.Normal("district_offset", 0, 1, dims="district")
    district_raw = district_offset * sigma_district
    district_effect = pm.Deterministic(
        "district_effect",
        district_raw - pm.math.mean(district_raw),
        dims="district"
    )

    sigma_daynight = pm.HalfNormal("sigma_daynight", sigma=0.5)
    daynight_offset = pm.Normal("daynight_offset", 0, 1, dims="daynight")
    daynight_raw = daynight_offset * sigma_daynight
    daynight_effect = pm.Deterministic(
        "daynight_effect",
        daynight_raw - pm.math.mean(daynight_raw),
        dims="daynight"
    )

    sigma_weather = pm.HalfNormal("sigma_weather", sigma=0.5)
    weather_offset = pm.Normal("weather_offset", 0, 1, dims="weather")
    weather_raw = weather_offset * sigma_weather
    weather_effect = pm.Deterministic(
        "weather_effect",
        weather_raw - pm.math.mean(weather_raw),
        dims="weather"
    )

    sigma_surface = pm.HalfNormal("sigma_surface", sigma=0.5)
    surface_offset = pm.Normal("surface_offset", 0, 1, dims="surface")
    surface_raw = surface_offset * sigma_surface
    surface_effect = pm.Deterministic(
        "surface_effect",
        surface_raw - pm.math.mean(surface_raw),
        dims="surface"
    )

    sigma_roadtype = pm.HalfNormal("sigma_roadtype", sigma=0.5)
    roadtype_offset = pm.Normal("roadtype_offset", 0, 1, dims="roadtype")
    roadtype_raw = roadtype_offset * sigma_roadtype
    roadtype_effect = pm.Deterministic(
        "roadtype_effect",
        roadtype_raw - pm.math.mean(roadtype_raw),
        dims="roadtype"
    )

    sigma_accident_type = pm.HalfNormal("sigma_accident_type", sigma=0.5)
    accident_type_offset = pm.Normal("accident_type_offset", 0, 1, dims="accident_type")
    accident_type_raw = accident_type_offset * sigma_accident_type
    accident_type_effect = pm.Deterministic(
        "accident_type_effect",
        accident_type_raw - pm.math.mean(accident_type_raw),
        dims="accident_type"
    )

    sigma_violation = pm.HalfNormal("sigma_violation", sigma=0.5)
    violation_offset = pm.Normal("violation_offset", 0, 1, dims="violation")
    violation_raw = violation_offset * sigma_violation
    violation_effect = pm.Deterministic(
        "violation_effect",
        violation_raw - pm.math.mean(violation_raw),
        dims="violation"
    )

    sigma_gender = pm.HalfNormal("sigma_gender", sigma=0.5)
    gender_offset = pm.Normal("gender_offset", 0, 1, dims="gender")
    gender_raw = gender_offset * sigma_gender
    gender_effect = pm.Deterministic(
        "gender_effect",
        gender_raw - pm.math.mean(gender_raw),
        dims="gender"
    )

    sigma_age = pm.HalfNormal("sigma_age", sigma=0.5)
    age_offset = pm.Normal("age_offset", 0, 1, dims="age")
    age_raw = age_offset * sigma_age
    age_effect = pm.Deterministic(
        "age_effect",
        age_raw - pm.math.mean(age_raw),
        dims="age"
    )

    sigma_month = pm.HalfNormal("sigma_month", sigma=0.5)
    month_offset = pm.Normal("month_offset", 0, 1, dims="month")
    month_raw = month_offset * sigma_month
    month_effect = pm.Deterministic(
        "month_effect",
        month_raw - pm.math.mean(month_raw),
        dims="month"
    )

    # 인덱스
    district_idx = train['District_idx'].values
    daynight_idx = train['주야_idx'].values
    weather_idx = train['기상상태_idx'].values
    surface_idx = train['노면상태_idx'].values
    roadtype_idx = train['도로형태_idx'].values
    accident_type_idx = train['사고유형_idx'].values
    violation_idx = train['법규위반_idx'].values
    gender_idx = train['가해운전자 성별_idx'].values
    age_idx = train['가해운전자 연령대_idx'].values
    month_idx = train['month_idx'].values

    eta = (
        alpha
        + district_effect[district_idx]
        + daynight_effect[daynight_idx]
        + weather_effect[weather_idx]
        + surface_effect[surface_idx]
        + roadtype_effect[roadtype_idx]
        + accident_type_effect[accident_type_idx]
        + violation_effect[violation_idx]
        + gender_effect[gender_idx]
        + age_effect[age_idx]
        + month_effect[month_idx]
    )

    mu = pm.Deterministic("mu", pm.math.sigmoid(eta))

    # Beta distribution parameterization
    a = mu * phi
    b = (1 - mu) * phi

    y_obs = pm.Beta(
        "y_obs",
        alpha=a,
        beta=b,
        observed=train_risk_beta
    )

    trace = pm.sample(
        draws=2000,
        tune=3000,
        chains=4,
        target_accept=0.97,
        random_seed=42
    )

 Progress                      Draw   Divergences   Step size   Grad evals   Speed           Elapsed   Remaining  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━   5000   0             0.049       127          15.94 draws/s   0:05:13   0:00:00    
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━   5000   2             0.044       127          7.31 draws/s    0:11:23   0:00:00    
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━   5000   0             0.036       127          4.76 draws/s    0:17:30   0:00:00    
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━   5000   1             0.047       63           3.54 draws/s    0:23:31   0:00:00

ERROR:pymc.stats.convergence:There were 3 divergences after tuning. Increase `target_accept` or reparameterize.


In [8]:
summary = az.summary(
    trace,
    var_names=[
        "alpha",
        "phi",
        "sigma_district",
        "sigma_daynight",
        "sigma_weather",
        "sigma_surface",
        "sigma_roadtype",
        "sigma_accident_type",
        "sigma_violation",
        "sigma_gender",
        "sigma_age",
        "sigma_month"
    ]
)

print(summary)

divergences = trace.sample_stats["diverging"].sum().values
print("Divergences:", divergences)

                      mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  \
alpha               -0.745  0.171  -1.093   -0.435      0.002    0.003   
phi                  2.812  0.123   2.589    3.048      0.001    0.001   
sigma_district       0.058  0.042   0.000    0.132      0.001    0.000   
sigma_daynight       0.187  0.211   0.000    0.593      0.003    0.003   
sigma_weather        0.146  0.135   0.000    0.390      0.002    0.002   
sigma_surface        0.182  0.177   0.000    0.511      0.003    0.003   
sigma_roadtype       0.089  0.079   0.000    0.222      0.001    0.002   
sigma_accident_type  0.957  0.189   0.630    1.311      0.003    0.002   
sigma_violation      0.091  0.082   0.000    0.236      0.001    0.002   
sigma_gender         0.164  0.182   0.000    0.513      0.003    0.003   
sigma_age            0.044  0.040   0.000    0.114      0.000    0.001   
sigma_month          0.061  0.044   0.000    0.135      0.001    0.001   

                     ess_bulk  ess_ta

In [9]:
# 6. Posterior sample 기반 test risk 예측

posterior = trace.posterior

def flatten_samples(x):
    return x.stack(sample=("chain", "draw")).transpose("sample", ...).values

alpha_s = flatten_samples(posterior["alpha"])
district_s = flatten_samples(posterior["district_effect"])
daynight_s = flatten_samples(posterior["daynight_effect"])
weather_s = flatten_samples(posterior["weather_effect"])
surface_s = flatten_samples(posterior["surface_effect"])
roadtype_s = flatten_samples(posterior["roadtype_effect"])
accident_type_s = flatten_samples(posterior["accident_type_effect"])
violation_s = flatten_samples(posterior["violation_effect"])
gender_s = flatten_samples(posterior["gender_effect"])
age_s = flatten_samples(posterior["age_effect"])
month_s = flatten_samples(posterior["month_effect"])

test_district = test['District_idx'].values
test_daynight = test['주야_idx'].values
test_weather = test['기상상태_idx'].values
test_surface = test['노면상태_idx'].values
test_roadtype = test['도로형태_idx'].values
test_accident_type = test['사고유형_idx'].values
test_violation = test['법규위반_idx'].values
test_gender = test['가해운전자 성별_idx'].values
test_age = test['가해운전자 연령대_idx'].values
test_month = test['month_idx'].values

eta_test = (
    alpha_s[:, None]
    + district_s[:, test_district]
    + daynight_s[:, test_daynight]
    + weather_s[:, test_weather]
    + surface_s[:, test_surface]
    + roadtype_s[:, test_roadtype]
    + accident_type_s[:, test_accident_type]
    + violation_s[:, test_violation]
    + gender_s[:, test_gender]
    + age_s[:, test_age]
    + month_s[:, test_month]
)

risk_test_samples = 1 / (1 + np.exp(-eta_test))

risk_mean = risk_test_samples.mean(axis=0) * 100
risk_lower = np.percentile(risk_test_samples, 2.5, axis=0) * 100
risk_upper = np.percentile(risk_test_samples, 97.5, axis=0) * 100

actual_score = test['injury_score'].values

In [10]:
# 7. 평가

mae = mean_absolute_error(actual_score, risk_mean)
rmse = np.sqrt(mean_squared_error(actual_score, risk_mean))
r2 = r2_score(actual_score, risk_mean)

print("\n--- Hierarchical Bayes 위험도 점수 예측 결과 ---")
print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

risk_result = test.copy()
risk_result['actual_score'] = actual_score
risk_result['predicted_risk_score'] = risk_mean
risk_result['risk_lower_95'] = risk_lower
risk_result['risk_upper_95'] = risk_upper

print("\n--- 실제 점수별 예측 위험도 평균 ---")
print(
    risk_result.groupby('actual_score')['predicted_risk_score']
    .agg(['count', 'mean', 'std'])
)

print("\n--- 예측 위험도 상위 20개 ---")
cols_to_show = [
    'predicted_risk_score',
    'risk_lower_95',
    'risk_upper_95',
    'actual_score',
    'District',
    '주야',
    '기상상태',
    '노면상태',
    '도로형태',
    '사고유형',
    '법규위반',
    '가해운전자 성별',
    '가해운전자 연령대'
]

print(
    risk_result.sort_values('predicted_risk_score', ascending=False)
    [cols_to_show]
    .head(20)
)


--- Hierarchical Bayes 위험도 점수 예측 결과 ---
MAE: 15.344400427215179
RMSE: 22.03622848769363
R2: 0.40255471500159945

--- 실제 점수별 예측 위험도 평균 ---
              count       mean        std
actual_score                             
0.0              75  11.240803   1.254786
25.0             13  33.407532  21.341006
50.0            105  45.756937  15.311227
75.0             51  41.510278  18.566696
100.0             1  52.185090        NaN

--- 예측 위험도 상위 20개 ---
      predicted_risk_score  risk_lower_95  risk_upper_95  actual_score  \
360              55.826879      49.633381      63.362455          50.0   
556              55.339121      49.525227      61.385015          50.0   
395              55.063013      48.868753      61.459896          50.0   
337              55.035756      49.527496      61.084649          50.0   
593              55.021546      49.072540      61.277562          75.0   
171              54.852167      46.348668      63.173278          75.0   
451              54.809406

In [11]:
# 8. 변수군별 효과 크기

sigma_summary = az.summary(
    trace,
    var_names=[
        "sigma_district",
        "sigma_daynight",
        "sigma_weather",
        "sigma_surface",
        "sigma_roadtype",
        "sigma_accident_type",
        "sigma_violation",
        "sigma_gender",
        "sigma_age",
        "sigma_month"
    ]
)

print("\n--- 변수군별 효과 크기 요약 ---")
print(sigma_summary)


--- 변수군별 효과 크기 요약 ---
                      mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  \
sigma_district       0.058  0.042    0.00    0.132      0.001    0.000   
sigma_daynight       0.187  0.211    0.00    0.593      0.003    0.003   
sigma_weather        0.146  0.135    0.00    0.390      0.002    0.002   
sigma_surface        0.182  0.177    0.00    0.511      0.003    0.003   
sigma_roadtype       0.089  0.079    0.00    0.222      0.001    0.002   
sigma_accident_type  0.957  0.189    0.63    1.311      0.003    0.002   
sigma_violation      0.091  0.082    0.00    0.236      0.001    0.002   
sigma_gender         0.164  0.182    0.00    0.513      0.003    0.003   
sigma_age            0.044  0.040    0.00    0.114      0.000    0.001   
sigma_month          0.061  0.044    0.00    0.135      0.001    0.001   

                     ess_bulk  ess_tail  r_hat  
sigma_district         3586.0    3725.0    1.0  
sigma_daynight         3098.0    4043.0    1.0  
sigma_weather  

In [12]:
# 9. 지역별 효과

district_effect_samples = district_s

district_result = pd.DataFrame({
    "District": encoders['District'].classes_,
    "district_effect_mean_logit": district_effect_samples.mean(axis=0),
    "lower_95": np.percentile(district_effect_samples, 2.5, axis=0),
    "upper_95": np.percentile(district_effect_samples, 97.5, axis=0)
})

district_result["relative_odds"] = np.exp(district_result["district_effect_mean_logit"])

district_result = district_result.sort_values(
    "district_effect_mean_logit",
    ascending=False
)

print("\n--- 지역별 위험도 효과 ---")
print(district_result)


--- 지역별 위험도 효과 ---
   District  district_effect_mean_logit  lower_95  upper_95  relative_odds
5       광진구                    0.041411 -0.069447  0.265328       1.042280
20      용산구                    0.023914 -0.097870  0.203956       1.024203
1       강동구                    0.021153 -0.086743  0.177750       1.021378
17      송파구                    0.018981 -0.070918  0.137136       1.019163
13     서대문구                    0.018801 -0.098901  0.184300       1.018979
19     영등포구                    0.015186 -0.102056  0.166268       1.015302
10     동대문구                    0.013118 -0.106183  0.165061       1.013205
24      중랑구                    0.009357 -0.109531  0.152539       1.009400
15      성동구                    0.008478 -0.127773  0.161989       1.008514
6       구로구                    0.004788 -0.122422  0.143376       1.004800
11      동작구                    0.002189 -0.127302  0.136720       1.002191
2       강북구                    0.001325 -0.135236  0.136218       1.001326
12   